# SynBioCrow 2.3 — 3D vs 2D Similarity Ablation (Development Split)\nUses only the 12-path Galaxy development split to compare Morgan/Tanimoto 2D similarity, USRCAT 3D similarity, and 3D-first with explicit 2D fallback. Candidate generation is not rerun. The 65-path benchmark split is not used for policy selection.\n

In [ ]:
import sys,subprocess,shutil,json,zipfile,hashlib,time
from pathlib import Path
from google.colab import drive, files

drive.mount("/content/drive",force_remount=False)

REPO=Path("/content/SynBioCrow_similarity_ablation")
SEALED=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/galaxy_development_sealed")
NORM=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/galaxy_literature_normalized")
OUT=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/similarity_ablation_development")
OUT.mkdir(parents=True,exist_ok=True)

def stream(cmd,cwd=None,timeout=3600):
    print("$"," ".join(map(str,cmd)),flush=True)
    p=subprocess.Popen(list(map(str,cmd)),cwd=cwd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,bufsize=1)
    t0=time.time(); lines=[]
    for line in p.stdout:
        print(line.rstrip(),flush=True); lines.append(line)
        if time.time()-t0>timeout:
            p.kill(); raise subprocess.TimeoutExpired(cmd,timeout)
    rc=p.wait()
    if rc:
        raise subprocess.CalledProcessError(rc,cmd,output="".join(lines))
    return rc

if REPO.exists():
    shutil.rmtree(REPO)

stream(["git","clone","--depth","1","--branch","develop/2.3",
        "https://github.com/theiman112860/SynBioCrow.git",str(REPO)],timeout=600)

stream([sys.executable,"-m","pip","install","-q","-e",str(REPO)],timeout=1200)

SCRIPT=REPO/"scripts/run_similarity_ablation_2_3.py"
stream([sys.executable,"-m","py_compile",str(SCRIPT)],timeout=120)
print("[PREFLIGHT] script syntax PASS",flush=True)

pred=SEALED/"sealed_predictions.json"
manifest=SEALED/"sealed_manifest.json"
bench=NORM/"galaxy_literature_benchmark_normalized.json"
targets=SEALED/"targets"

for p in (pred,manifest,bench,targets):
    assert p.exists(),p

m=json.loads(manifest.read_text())
actual=hashlib.sha256(pred.read_bytes()).hexdigest()
assert actual==m["predictions_sha256"],(actual,m["predictions_sha256"])
print("[PREFLIGHT] sealed predictions verified",actual,flush=True)

stream([sys.executable,str(SCRIPT),
        "--targets-dir",str(targets),
        "--manifest",str(manifest),
        "--predictions",str(pred),
        "--benchmark",str(bench),
        "--output-dir",str(OUT),
        "--seed","20230930"],cwd=REPO,timeout=3600)

bundle=OUT/"SynBioCrow_2_3_SIMILARITY_ABLATION_DEVELOPMENT_RESULTS.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*.json")):
        z.write(p,str(p.relative_to(OUT)))
    z.write(SCRIPT,"run_similarity_ablation_2_3.py")

print("BUNDLE",bundle)
print("SHA256",hashlib.sha256(bundle.read_bytes()).hexdigest())
files.download(str(bundle))
